# Colab Notebook for SAM2 fine-tuning

Fine-tune SAM 2.1 Base+ or Large on annotated frames from your videos, then compare pretrained and fine-tuned weights on held-out videos.<br>
Use a **GPU** runtime and run the cells in order. This is single-object, single-frame image fine-tuning; temporal memory stays pretrained.<br>
ref: [SAM 2](https://github.com/facebookresearch/sam2) and its official training model and loss.

## Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import sys
import subprocess
from pathlib import Path

UPSTREAM_SHA = '2b90b9f5ceec907a1c18123530e92e794ad901a4'
SOURCE_DIR = Path('/content') / f'sam2-ft-{UPSTREAM_SHA[:12]}'
if not SOURCE_DIR.exists():
    subprocess.run(['git', 'clone', 'https://github.com/facebookresearch/sam2.git',
                    str(SOURCE_DIR)], check=True)
    subprocess.run(['git', '-C', str(SOURCE_DIR), 'checkout', '--detach', UPSTREAM_SHA],
                   check=True)
revision = subprocess.check_output(
    ['git', '-C', str(SOURCE_DIR), 'rev-parse', 'HEAD'], text=True).strip()
if revision != UPSTREAM_SHA:
    raise RuntimeError(f'Expected SAM2 revision {UPSTREAM_SHA}, found {revision}')

# Video comparison disables optional CUDA-extension postprocessing.
install_env = dict(os.environ, SAM2_BUILD_CUDA='0')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-build-isolation',
                '-e', str(SOURCE_DIR), 'fvcore', 'tensordict>=0.6.0',
                'tensorboard', 'opencv-python', 'matplotlib'],
               env=install_env, check=True)
os.chdir(SOURCE_DIR)
sys.path.insert(0, str(SOURCE_DIR))

### Import libraries

Start with a fresh GPU runtime. If installation requests a runtime restart, restart and run from Setup again.

In [ ]:
import copy
import csv
import gc
import json
import math
import random
import re
import shutil
import tempfile
import uuid
from collections import defaultdict
from datetime import datetime, timezone
from functools import partial
from urllib.request import urlretrieve

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader
from hydra import compose
from hydra.utils import instantiate
from omegaconf import OmegaConf, open_dict
from tqdm.auto import tqdm

import sam2
from sam2.build_sam import build_sam2_video_predictor
from sam2.sam2_image_predictor import SAM2ImagePredictor
from sam2.utils.transforms import SAM2Transforms
from training.utils.data_utils import Frame, Object, VideoDatapoint, collate_fn
from training.trainer import CORE_LOSS_KEY

if not torch.cuda.is_available():
    raise RuntimeError('Select Runtime > Change runtime type > GPU in Colab.')
DEVICE = torch.device('cuda')
AMP_DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
plt.rcParams.update({'figure.figsize': (6, 4), 'font.size': 11})
print('PyTorch:', torch.__version__, '| GPU:', torch.cuda.get_device_name())
print('Mixed precision:', AMP_DTYPE)

## Prepare data

Set paths and parameters below. Put JPEG frames in `images/<video_id>/` and binary PNG masks in `masks/<video_id>/`:

    finetune_data/
      images/video_A/frame2.jpg
      images/video_A/frame10.jpeg
      images/video_B/another_name.jpg
      masks/video_A/frame2.png
      masks/video_B/another_name.png

A mask uses the same stem as its image; 0 is background and 1 or 255 is foreground.
Video lengths, image sizes, and source names may vary. Unlabeled frames remain in video evaluation.
Orphan masks are reported and skipped. Empty masks are kept for video evaluation and excluded from positive-point image training/validation.<br>
Split by video ID before selecting frames. Validation selects the best checkpoint; test videos are used only for the final comparison.

In [ ]:
WORK_DIR = Path('/content/drive/MyDrive/Colab Notebooks/SAM2-segmentation')
DATA_ROOT = WORK_DIR / 'finetune_data'
RUNS_ROOT = WORK_DIR / 'finetune_runs'
CACHE_DATA = False  # Optional local copy; the source on Drive is preserved.

SEED = 42
SPLIT_RATIOS = {'train': 0.6, 'val': 0.2, 'test': 0.2}
EPOCHS = 20
BATCH_SIZE = 1
LEARNING_RATE = 5e-6
WEIGHT_DECAY = 0.1
MAX_GRAD_NORM = 0.1
FREEZE_IMAGE_ENCODER = True
CORRECTION_POINTS = 1  # Extra correction clicks during training only.
SAVE_LAST = False  # Optional optimizer/scaler checkpoint.

MODEL_SIZE = 'base_plus'  # Change to 'large' to train and compare Large.
MODEL_FILES = {
    'base_plus': ('sam2.1_hiera_base_plus.pt', 'configs/sam2.1/sam2.1_hiera_b+.yaml'),
    'large': ('sam2.1_hiera_large.pt', 'configs/sam2.1/sam2.1_hiera_l.yaml'),
}
checkpoint_filename, INFERENCE_CONFIG = MODEL_FILES[MODEL_SIZE]
PRETRAINED_PATH = WORK_DIR / 'checkpoints' / checkpoint_filename
LOSS_CONFIG_SOURCE = 'configs/sam2.1_training/sam2.1_hiera_b+_MOSE_finetune.yaml'
IMAGE_SIZE = 1024

### Prepare functions

In [ ]:
def natural_key(value):
    parts = tuple((1, int(part)) if part.isdigit() else (0, part.casefold())
                  for part in re.split(r'(\d+)', str(value)))
    return parts, str(value)


def read_mask(path):
    with Image.open(path) as image:
        mask = np.asarray(image)
    # Accept grayscale RGB exports, but do not merge differently colored objects.
    if mask.ndim == 3 and mask.shape[2] in (3, 4):
        if not np.all(mask[..., :3] == mask[..., :1]):
            raise ValueError(f'Use a single-object binary grayscale mask: {path}')
        mask = mask[..., 0]
    if mask.ndim != 2:
        raise ValueError(f'Expected a 2D mask: {path}')
    values = set(np.unique(mask).tolist())
    if not (values <= {0, 1} or values <= {0, 255}):
        raise ValueError(f'Mask must contain 0/1 or 0/255: {path}')
    return mask != 0


def scan_dataset(root):
    image_root, mask_root = root / 'images', root / 'masks'
    if not image_root.is_dir():
        raise ValueError(f'Create images/<video_id>/ under DATA_ROOT: {root}')
    masks = {}
    if mask_root.is_dir():
        for path in mask_root.glob('*/*'):
            if path.is_file() and path.suffix.lower() == '.png':
                key = (path.parent.name, path.stem)
                if key in masks:
                    raise ValueError(f'Duplicate mask stem: {key}')
                masks[key] = path
    records, used_masks, seen = [], set(), set()
    for path in sorted(image_root.glob('*/*'), key=lambda item: natural_key(item.as_posix())):
        if not path.is_file() or path.suffix.lower() not in {'.jpg', '.jpeg'}:
            continue
        key = (path.parent.name, path.stem)
        if key in seen:
            raise ValueError(f'Duplicate image stem (jpg/jpeg): {key}')
        seen.add(key)
        with Image.open(path) as image:
            width, height = image.size
        mask_path = masks.get(key)
        foreground = False
        if mask_path is not None:
            mask = read_mask(mask_path)
            if mask.shape != (height, width):
                raise ValueError(
                    f'Image/mask shape mismatch for {path}: '
                    f'image {(height, width)}, mask {mask.shape}')
            foreground = bool(mask.any())
            used_masks.add(key)
        records.append({
            'video_id': key[0], 'stem': key[1],
            'image': path.relative_to(root).as_posix(),
            'mask': mask_path.relative_to(root).as_posix() if mask_path else None,
            'width': width, 'height': height, 'foreground': foreground,
        })
    if not records:
        raise ValueError(f'No JPEG frames found under {image_root}/<video_id>/')
    orphans = sorted(set(masks) - used_masks, key=lambda key: natural_key('/'.join(key)))
    print(f'Orphan masks skipped: {len(orphans)}')
    for key in orphans[:10]:
        print(' ', masks[key].relative_to(root).as_posix())
    if len(orphans) > 10:
        print(' ...')
    return records

In [ ]:
def split_videos(video_ids, ratios, seed):
    names = ['train', 'val', 'test']
    fractions = np.array([ratios[name] for name in names], dtype=float)
    if (not np.isfinite(fractions).all() or (fractions < 0).any()
            or not np.isclose(fractions.sum(), 1)
            or fractions[0] <= 0 or fractions[1] <= 0):
        raise ValueError('Split ratios must sum to 1; train and val must be positive.')
    shuffled = sorted(video_ids, key=natural_key)
    random.Random(seed).shuffle(shuffled)
    active = np.flatnonzero(fractions > 0)
    if len(shuffled) < len(active):
        raise ValueError('Not enough labeled video IDs for the requested nonempty splits. '
                         'Add videos or set the test ratio to 0 and adjust train/val.')
    expected = fractions * len(shuffled)
    counts = np.floor(expected).astype(int)
    for index in sorted(active, key=lambda index: -(expected[index] - counts[index]))[
            :len(shuffled) - int(counts.sum())]:
        counts[index] += 1
    for index in active:
        if counts[index] == 0:
            donor = max((item for item in active if counts[item] > 1),
                        key=lambda item: counts[item] - expected[item])
            counts[donor] -= 1
            counts[index] += 1
    result, start = {}, 0
    for name, count in zip(names, counts):
        result[name] = shuffled[start:start + count]
        start += count
    return result


def write_csv(path, rows, fieldnames):
    with Path(path).open('w', encoding='utf-8-sig', newline='') as stream:
        writer = csv.DictWriter(stream, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)


def binary_metrics(prediction, target):
    prediction, target = np.asarray(prediction, dtype=bool), np.asarray(target, dtype=bool)
    if prediction.shape != target.shape:
        raise ValueError(f'Metric shape mismatch: {prediction.shape} vs {target.shape}')
    intersection = int(np.logical_and(prediction, target).sum())
    predicted_area, target_area = int(prediction.sum()), int(target.sum())
    union = predicted_area + target_area - intersection
    return {
        'dice': 2 * intersection / (predicted_area + target_area)
                if predicted_area + target_area else 1.0,
        'iou': intersection / union if union else 1.0,
    }


def macro_by_video(frame_rows):
    grouped = defaultdict(list)
    for row in frame_rows:
        grouped[row['video_id']].append(row)
    return [
        {'video_id': video_id, 'frames': len(rows),
         'dice': float(np.mean([row['dice'] for row in rows])),
         'iou': float(np.mean([row['iou'] for row in rows]))}
        for video_id, rows in sorted(grouped.items(), key=lambda item: natural_key(item[0]))
    ]


def fixed_positive_point(mask):
    coordinates = np.argwhere(mask)
    if len(coordinates) == 0:
        raise ValueError('A positive prompt requires a nonempty mask.')
    center = coordinates.mean(axis=0)
    index = np.argmin(((coordinates - center) ** 2).sum(axis=1))
    y_coord, x_coord = coordinates[index]
    return [float(x_coord), float(y_coord)]

### Split videos and save the manifest

In [ ]:
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

records = scan_dataset(DATA_ROOT)
labeled_videos = {row['video_id'] for row in records if row['mask'] is not None}
splits = split_videos(labeled_videos, SPLIT_RATIOS, SEED)
video_split = {video_id: split for split, videos in splits.items() for video_id in videos}
for row in records:
    row['split'] = video_split.get(row['video_id'], 'unlabeled')

train_records = [row for row in records if row['split'] == 'train' and row['foreground']]
val_records = [row for row in records if row['split'] == 'val' and row['foreground']]
if not train_records or not val_records:
    raise ValueError('Train and val each need labeled nonempty masks for positive-point '
                     'image learning. Adjust the seed/ratios or add annotations.')

empty_count = sum(row['mask'] is not None and not row['foreground'] for row in records)
print('Empty masks excluded from image training/validation:', empty_count)
print('Videos without labels (not split):',
      len({row['video_id'] for row in records} - labeled_videos))
for split, videos in splits.items():
    print(f'{split}: {len(videos)} videos, '
          f'{sum(row["mask"] is not None and row["split"] == split for row in records)} labels')
print('Positive training/validation frames:', len(train_records), '/', len(val_records))

run_id = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '_' + uuid.uuid4().hex[:8]
RUN_DIR = RUNS_ROOT / run_id
RUN_DIR.mkdir(parents=True, exist_ok=False)
manifest = {'seed': SEED, 'ratios': SPLIT_RATIOS, 'splits': splits, 'frames': records}
(RUN_DIR / 'manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')

ACTIVE_DATA_ROOT = DATA_ROOT
if CACHE_DATA:
    ACTIVE_DATA_ROOT = Path(tempfile.mkdtemp(prefix='sam2-data-', dir='/content'))
    shutil.copytree(DATA_ROOT, ACTIVE_DATA_ROOT, dirs_exist_ok=True)
print('Run directory:', RUN_DIR)

## Load model

Use the selected official model architecture and its full pretrained state. Only the prompt encoder and mask decoder are trained by default.
The image encoder can be unfrozen with `FREEZE_IMAGE_ENCODER = False`; temporal memory and object-pointer parameters stay frozen.<br>
Inputs use the same 1024-square bilinear resize and ImageNet normalization as the SAM2 image predictor, with nearest-neighbor masks. No intensity correction or color augmentation is applied.<br>
The official video loader uses its own PIL resize during the final video comparison.

In [ ]:
PRETRAINED_PATH.parent.mkdir(parents=True, exist_ok=True)
if not PRETRAINED_PATH.exists():
    urlretrieve(
        f'https://dl.fbaipublicfiles.com/segment_anything_2/092824/{checkpoint_filename}',
        PRETRAINED_PATH)

# Use the training YAML for the shared loss, and the selected inference
# YAML for model architecture. Do not resolve the unused Trainer subtree.
cfg = compose(config_name=LOSS_CONFIG_SOURCE)
model_cfg = copy.deepcopy(compose(config_name=INFERENCE_CONFIG).model)
with open_dict(model_cfg):
    model_cfg._target_ = 'training.model.sam2.SAM2Train'
    model_cfg.image_size = IMAGE_SIZE
    model_cfg.freeze_image_encoder = FREEZE_IMAGE_ENCODER
    model_cfg.prob_to_use_pt_input_for_train = 1.0
    model_cfg.prob_to_use_pt_input_for_eval = 1.0
    model_cfg.prob_to_use_box_input_for_train = 0.0
    model_cfg.prob_to_use_box_input_for_eval = 0.0
    model_cfg.prob_to_sample_from_gt_for_train = 0.1
    model_cfg.add_all_frames_to_correct_as_cond = True
    model_cfg.num_frames_to_correct_for_train = 1
    model_cfg.num_frames_to_correct_for_eval = 1
    model_cfg.num_init_cond_frames_for_train = 1
    model_cfg.num_init_cond_frames_for_eval = 1
    model_cfg.rand_frames_to_correct_for_train = False
    model_cfg.rand_init_cond_frames_for_train = False
    model_cfg.num_correction_pt_per_frame = CORRECTION_POINTS
    model_cfg.pt_sampling_for_eval = 'center'
    model_cfg.forward_backbone_per_frame_for_eval = False

model = instantiate(model_cfg, _recursive_=True)
pretrained = torch.load(PRETRAINED_PATH, map_location='cpu', weights_only=True)
model.load_state_dict(pretrained['model'], strict=True)
del pretrained
for name, parameter in model.named_parameters():
    parameter.requires_grad = (
        name.startswith(('sam_prompt_encoder.', 'sam_mask_decoder.'))
        or (not FREEZE_IMAGE_ENCODER and name.startswith('image_encoder.'))
    )
model = model.to(DEVICE)
criterion = instantiate(cfg.trainer.loss.all).to(DEVICE)
trainable_parameters = [parameter for parameter in model.parameters() if parameter.requires_grad]
optimizer = torch.optim.AdamW(trainable_parameters, lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scaler = torch.amp.GradScaler('cuda', enabled=AMP_DTYPE == torch.float16)
print('Trainable parameters:', sum(parameter.numel() for parameter in trainable_parameters))

### Prepare image batches

In [ ]:
image_transform = SAM2Transforms(resolution=IMAGE_SIZE, mask_threshold=0.0)
video_numbers = {video_id: index for index, video_id in enumerate(sorted(labeled_videos))}


class ImageFrames(Dataset):
    def __init__(self, frame_records, root):
        self.records = frame_records
        self.root = root

    def __len__(self):
        return len(self.records)

    def __getitem__(self, index):
        row = self.records[index]
        with Image.open(self.root / row['image']) as image:
            image_tensor = image_transform(image.convert('RGB'))
        mask = read_mask(self.root / row['mask'])
        resized_mask = np.asarray(Image.fromarray(mask).resize(
            (IMAGE_SIZE, IMAGE_SIZE), resample=Image.Resampling.NEAREST)).copy()
        if not resized_mask.any():
            raise ValueError(f'Foreground disappeared at 1024-square mask resize: {row["mask"]}. '
                             'The resized mask has no foreground pixels and cannot supply a positive training point.')
        frame = Frame(image_tensor, [Object(1, 0, torch.from_numpy(resized_mask))])
        return VideoDatapoint([frame], video_numbers[row['video_id']],
                              (row['height'], row['width']))


train_dataset = ImageFrames(train_records, ACTIVE_DATA_ROOT)
val_dataset = ImageFrames(val_records, ACTIVE_DATA_ROOT)
train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0,
    collate_fn=partial(collate_fn, dict_key='all'),
    generator=torch.Generator().manual_seed(SEED))

### Prepare validation

Use the official SAM2 image predictor with the same model, without creating another GPU model.
Each validation image receives **one fixed positive point** from its mask and **zero correction clicks**.
Choose the candidate with the highest predicted IoU score, without using ground truth to select masks.
The image predictor returns binary masks at the original image size for Dice/IoU scoring.
Average frames within each video, then average videos equally. A pair of empty masks scores 1.

In [ ]:
def float_outputs(value):
    if torch.is_tensor(value):
        return value.float() if value.is_floating_point() else value
    if isinstance(value, dict):
        return {key: float_outputs(item) for key, item in value.items()}
    if isinstance(value, list):
        return [float_outputs(item) for item in value]
    if isinstance(value, tuple):
        return tuple(float_outputs(item) for item in value)
    return value


@torch.no_grad()
def validate_images(model, dataset):
    was_training = model.training
    predictor = SAM2ImagePredictor(model)
    frame_rows = []
    try:
        model.eval()
        for row in tqdm(dataset.records, desc='Validation'):
            with Image.open(dataset.root / row['image']) as image:
                image_rgb = np.asarray(image.convert('RGB'))
            target = read_mask(dataset.root / row['mask'])
            point = fixed_positive_point(target)
            with torch.autocast('cuda', dtype=AMP_DTYPE):
                predictor.set_image(image_rgb)
                masks, scores, _ = predictor.predict(
                    point_coords=np.array([point], dtype=np.float32),
                    point_labels=np.array([1], dtype=np.int32),
                    multimask_output=True)
            prediction = masks[int(np.argmax(scores))].astype(bool)
            frame_rows.append({'video_id': row['video_id'], 'image': row['image'],
                               **binary_metrics(prediction, target)})
        video_rows = macro_by_video(frame_rows)
        metrics = {key: float(np.mean([row[key] for row in video_rows]))
                   for key in ('dice', 'iou')}
        return metrics, frame_rows, video_rows
    finally:
        predictor.reset_predictor()
        model.train(was_training)
        if FREEZE_IMAGE_ENCODER:
            model.image_encoder.eval()

## Train

In [ ]:
run_config = {
    'upstream_repository': 'https://github.com/facebookresearch/sam2',
    'upstream_sha': UPSTREAM_SHA, 'model_size': MODEL_SIZE,
    'loss_config_source': LOSS_CONFIG_SOURCE, 'inference_config': INFERENCE_CONFIG,
    'pretrained_checkpoint': PRETRAINED_PATH.name, 'seed': SEED,
    'epochs': EPOCHS, 'batch_size': BATCH_SIZE, 'learning_rate': LEARNING_RATE,
    'weight_decay': WEIGHT_DECAY, 'max_grad_norm': MAX_GRAD_NORM,
    'freeze_image_encoder': FREEZE_IMAGE_ENCODER,
    'training_correction_points': CORRECTION_POINTS,
    'validation_correction_points': 0, 'validation_prompt': 'one fixed positive point',
    'validation_api': 'SAM2ImagePredictor',
    'validation_mask_selection': 'highest_predicted_iou',
    'image_size': IMAGE_SIZE, 'amp_dtype': str(AMP_DTYPE),
    'torch_version': torch.__version__, 'data_root': str(DATA_ROOT),
    'cache_data': CACHE_DATA, 'video_fill_hole_area': 0,
    'video_dynamic_multimask': True, 'video_binarize_prompt_masks': True,
    'best_selection': 'macro_per_video_validation_dice',
}
(RUN_DIR / 'run_config.json').write_text(json.dumps(run_config, indent=2), encoding='utf-8')
(RUN_DIR / 'model_config.yaml').write_text(
    OmegaConf.to_yaml(model_cfg, resolve=True), encoding='utf-8')
(RUN_DIR / 'loss_config.yaml').write_text(
    OmegaConf.to_yaml(cfg.trainer.loss.all, resolve=True), encoding='utf-8')

BEST_PATH = RUN_DIR / 'best.pt'
history, best_dice = [], -math.inf
for epoch in range(1, EPOCHS + 1):
    model.train()
    if FREEZE_IMAGE_ENCODER:
        model.image_encoder.eval()
    loss_sum, image_count = 0.0, 0
    progress = tqdm(train_loader, desc=f'Epoch {epoch}/{EPOCHS}')
    for batch in progress:
        batch = batch.to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast('cuda', dtype=AMP_DTYPE):
            outputs = model(batch)
        # Official multi-step focal + Dice + IoU + object-score loss in float32.
        with torch.autocast('cuda', enabled=False):
            losses = criterion(float_outputs(outputs), batch.masks)
            loss = losses[CORE_LOSS_KEY]
        if not torch.isfinite(loss):
            raise RuntimeError(f'Non-finite training loss at epoch {epoch}.')
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(trainable_parameters, MAX_GRAD_NORM)
        scaler.step(optimizer)
        scaler.update()
        batch_images = batch.num_videos
        loss_sum += float(loss.detach()) * batch_images
        image_count += batch_images
        progress.set_postfix(loss=loss_sum / image_count)
        del outputs, losses, loss, batch

    metrics, val_frames, val_videos = validate_images(model, val_dataset)
    history.append({'epoch': epoch, 'train_loss': loss_sum / image_count,
                    'val_dice': metrics['dice'], 'val_iou': metrics['iou']})
    write_csv(RUN_DIR / 'history.csv', history, ['epoch', 'train_loss', 'val_dice', 'val_iou'])
    if metrics['dice'] > best_dice:
        best_dice = metrics['dice']
        # Full CPU weights remain compatible with build_sam2_video_predictor.
        cpu_weights = {name: value.detach().cpu() for name, value in model.state_dict().items()}
        torch.save({'model': cpu_weights}, BEST_PATH)
        del cpu_weights
        best_metadata = {**run_config, 'best_epoch': epoch, 'validation': metrics,
                         'checkpoint': BEST_PATH.name}
        (RUN_DIR / 'best_metadata.json').write_text(
            json.dumps(best_metadata, indent=2), encoding='utf-8')
        write_csv(RUN_DIR / 'best_validation_frames.csv', val_frames,
                  ['video_id', 'image', 'dice', 'iou'])
        write_csv(RUN_DIR / 'best_validation_videos.csv', val_videos,
                  ['video_id', 'frames', 'dice', 'iou'])
    if SAVE_LAST:
        torch.save({'model': {name: value.detach().cpu()
                              for name, value in model.state_dict().items()},
                    'optimizer': optimizer.state_dict(), 'scaler': scaler.state_dict(),
                    'epoch': epoch, 'history': history, 'run_config': run_config},
                   RUN_DIR / 'last.pt')
    print(f'Epoch {epoch}: loss={history[-1]["train_loss"]:.4f}, '
          f'val Dice={metrics["dice"]:.4f}, val IoU={metrics["iou"]:.4f}')

print('Best inference checkpoint:', BEST_PATH)
print('Inference config:', INFERENCE_CONFIG)

### Training curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
epochs = [row['epoch'] for row in history]
axes[0].plot(epochs, [row['train_loss'] for row in history], color='#2563a6')
axes[0].set(xlabel='Epoch', ylabel='Official training loss', title='Training')
axes[1].plot(epochs, [row['val_dice'] for row in history], label='Dice', color='#2563a6')
axes[1].plot(epochs, [row['val_iou'] for row in history], '--', label='IoU', color='#c06b19')
axes[1].set(xlabel='Epoch', ylabel='Macro per-video score', ylim=(0, 1),
            title='Validation: one click, no corrections')
axes[1].legend()
for axis in axes:
    axis.grid(alpha=0.2)
fig.tight_layout()
fig.savefig(RUN_DIR / 'training_curves.png', dpi=160)
plt.show()

## Evaluate videos

Compare **pretrained** and **fine-tuned** weights of the selected model with identical prompts on test videos.
Naturally sort all JPEG frames, including unlabeled frames, and copy their bytes to temporary numeric filenames for the official video loader. Save the source-to-index map.<br>
Use one fixed positive point from the first labeled nonempty mask. Propagate **forward only**, with no later ground-truth clicks.
Score labeled frames strictly after the prompted frame, including empty masks; earlier frames and the prompted frame are excluded.
Report frame counts and videos with no evaluable propagated labels. Frames must share dimensions within each tested video; training images may have different dimensions.<br>
Only one model is on the GPU at a time. Video frames and inference state are offloaded to CPU.<br>
Both predictors keep the inference notebook's dynamic multimask fallback and prompt-mask binarization. CUDA-extension hole filling is disabled (fill_hole_area=0) in this comparison.<br>
Save original-size binary PNG masks (0/255) for every forward-propagated frame, including the prompt and unlabeled frames, under the run directory in video_masks/. Metrics still exclude the prompt and use labeled frames only.

In [ ]:
# Release training memory before creating the video predictor.
del optimizer, scaler, criterion, trainable_parameters, model
gc.collect()
torch.cuda.empty_cache()


test_sequences, protocol = {}, {}
for video_id in splits['test']:
    sequence = sorted([row for row in records if row['video_id'] == video_id],
                      key=lambda row: natural_key(Path(row['image']).name))
    sizes = {(row['height'], row['width']) for row in sequence}
    if len(sizes) != 1:
        raise ValueError(f'Test video {video_id} has mixed frame sizes: {sorted(sizes)}. '
                         'Prepare a consistently sized sequence for video propagation.')
    prompt_index = next((index for index, row in enumerate(sequence)
                         if row['foreground']), None)
    point = (fixed_positive_point(read_mask(ACTIVE_DATA_ROOT / sequence[prompt_index]['mask']))
             if prompt_index is not None else None)
    test_sequences[video_id] = sequence
    protocol[video_id] = {
        'prompt_index': prompt_index, 'point_xy': point,
        'direction': 'forward', 'exclude_prompt_frame': True,
        'source_to_index': [{'index': index, 'image': row['image'], 'mask': row['mask']}
                            for index, row in enumerate(sequence)],
    }
(RUN_DIR / 'video_protocol.json').write_text(
    json.dumps(protocol, indent=2), encoding='utf-8')

In [ ]:
def make_video_predictor(checkpoint):
    return build_sam2_video_predictor(
        INFERENCE_CONFIG, str(checkpoint), device=str(DEVICE), apply_postprocessing=False,
        hydra_overrides_extra=[
            '++model.sam_mask_decoder_extra_args.dynamic_multimask_via_stability=true',
            '++model.sam_mask_decoder_extra_args.dynamic_multimask_stability_delta=0.05',
            '++model.sam_mask_decoder_extra_args.dynamic_multimask_stability_thresh=0.98',
            '++model.binarize_mask_from_pts_for_mem_enc=true',
            '++model.fill_hole_area=0',
        ])


@torch.inference_mode()
def save_video_masks(predictor, sequence, prompt, mask_dir):
    mask_dir.mkdir(parents=True, exist_ok=True)
    state = None
    with tempfile.TemporaryDirectory(prefix='sam2-video-', dir='/content') as temp_dir:
        for index, row in enumerate(sequence):
            shutil.copyfile(ACTIVE_DATA_ROOT / row['image'],
                            Path(temp_dir) / f'{index:08d}.jpg')
        try:
            with torch.autocast('cuda', dtype=AMP_DTYPE):
                state = predictor.init_state(
                    video_path=temp_dir, offload_video_to_cpu=True, offload_state_to_cpu=True)
                predictor.add_new_points_or_box(
                    inference_state=state, frame_idx=prompt['prompt_index'], obj_id=1,
                    points=np.array([prompt['point_xy']], dtype=np.float32),
                    labels=np.array([1], dtype=np.int32))
                for frame_index, object_ids, logits in predictor.propagate_in_video(
                        state, start_frame_idx=prompt['prompt_index'], reverse=False):
                    object_index = list(object_ids).index(1)
                    prediction = (logits[object_index, 0] > 0).cpu().numpy()
                    Image.fromarray(prediction.astype(np.uint8) * 255).save(
                        mask_dir / f'{frame_index:08d}.png')
        finally:
            if state is not None:
                predictor.reset_state(state)


def evaluate_video(predictor, model_name, video_id, sequence, prompt):
    prompt_index = prompt['prompt_index']
    if prompt_index is None:
        return [], 'no_nonempty_prompt_mask'
    evaluable = {index: row for index, row in enumerate(sequence)
                 if index > prompt_index and row['mask'] is not None}
    mask_dir = RUN_DIR / 'video_masks' / model_name / video_id
    save_video_masks(predictor, sequence, prompt, mask_dir)
    frame_rows = []
    for frame_index, row in evaluable.items():
        with Image.open(mask_dir / f'{frame_index:08d}.png') as mask_image:
            prediction = np.asarray(mask_image) != 0
        target = read_mask(ACTIVE_DATA_ROOT / row['mask'])
        frame_rows.append({'video_id': video_id, 'frame_index': frame_index,
                           'image': row['image'], **binary_metrics(prediction, target)})
    return frame_rows, 'ok' if evaluable else 'no_labeled_frames_after_prompt'

In [ ]:
comparison_frames, comparison_videos, comparison_summary = [], [], []
if not test_sequences:
    print('Test ratio is 0: held-out video comparison skipped.')
else:
    for model_name, checkpoint in [('pretrained', PRETRAINED_PATH), ('fine_tuned', BEST_PATH)]:
        predictor = make_video_predictor(checkpoint)
        model_frames = []
        for video_id, sequence in tqdm(test_sequences.items(), desc=model_name):
            frame_rows, status = evaluate_video(predictor, model_name, video_id, sequence, protocol[video_id])
            model_frames.extend(frame_rows)
            video_result = {
                'model': model_name, 'video_id': video_id, 'status': status,
                'frames': len(frame_rows),
                'dice': float(np.mean([row['dice'] for row in frame_rows])) if frame_rows else None,
                'iou': float(np.mean([row['iou'] for row in frame_rows])) if frame_rows else None,
            }
            comparison_videos.append(video_result)
            if status != 'ok':
                print(video_id, ':', status)
        comparison_frames.extend({'model': model_name, **row} for row in model_frames)
        video_rows = macro_by_video(model_frames)
        summary = {'model': model_name, 'videos': len(video_rows),
                   'frames': len(model_frames),
                   'dice': float(np.mean([row['dice'] for row in video_rows])) if video_rows else None,
                   'iou': float(np.mean([row['iou'] for row in video_rows])) if video_rows else None}
        comparison_summary.append(summary)
        print(summary)
        del predictor
        gc.collect()
        torch.cuda.empty_cache()

write_csv(RUN_DIR / 'test_video_frames.csv', comparison_frames,
          ['model', 'video_id', 'frame_index', 'image', 'dice', 'iou'])
write_csv(RUN_DIR / 'test_videos.csv', comparison_videos,
          ['model', 'video_id', 'status', 'frames', 'dice', 'iou'])
write_csv(RUN_DIR / 'test_summary.csv', comparison_summary,
          ['model', 'videos', 'frames', 'dice', 'iou'])

### Compare results

In [ ]:
scored = [row for row in comparison_summary if row['videos'] > 0]
if scored:
    positions = np.arange(len(scored))
    fig, axis = plt.subplots(figsize=(6, 4))
    axis.bar(positions - 0.18, [row['dice'] for row in scored], width=0.36,
             label='Dice', color='#2563a6')
    axis.bar(positions + 0.18, [row['iou'] for row in scored], width=0.36,
             label='IoU', color='#c06b19', hatch='//')
    axis.set_xticks(positions, [row['model'] for row in scored])
    axis.set(ylim=(0, 1), ylabel='Macro per-video score',
             title=f'Held-out video propagation ({MODEL_SIZE})')
    axis.legend()
    fig.tight_layout()
    fig.savefig(RUN_DIR / 'test_comparison.png', dpi=160)
    plt.show()
else:
    print('No evaluable held-out propagated labels; see test_videos.csv for the reasons.')

### Compare segmentation masks

Show the original image beside pretrained and fine-tuned predictions, using the same red overlay and outline.
Choose a test video and optional frame indices below. By default, show up to three frames evenly spaced in time after the prompt.
The display does not adjust image intensity or select frames by score.<br>
Cached masks are reused. If masks are missing, only the selected video is propagated again, one model at a time; training is not repeated.
The figure is saved in the run directory as `segmentation_comparison_<video_id>.png`.

In [ ]:
def ensure_comparison_masks(video_id, sequence, prompt, frame_indices):
    for model_name, checkpoint in [('pretrained', PRETRAINED_PATH), ('fine_tuned', BEST_PATH)]:
        mask_dir = RUN_DIR / 'video_masks' / model_name / video_id
        if all((mask_dir / f'{index:08d}.png').is_file() for index in frame_indices):
            continue
        print(f'Caching {model_name} masks for {video_id}...')
        predictor = make_video_predictor(checkpoint)
        try:
            save_video_masks(predictor, sequence, prompt, mask_dir)
        finally:
            del predictor
            gc.collect()
            torch.cuda.empty_cache()


def show_segmentation_comparison(video_id, frame_indices=None):
    if video_id is None:
        print('No test video is available. Set a positive test split to compare masks.')
        return None
    if video_id not in test_sequences:
        raise ValueError(f'Choose VIDEO_TO_SHOW from: {list(test_sequences)}')
    sequence, prompt = test_sequences[video_id], protocol[video_id]
    prompt_index = prompt['prompt_index']
    if prompt_index is None:
        print(f'{video_id}: no nonempty initial prompt mask is available.')
        return None
    if frame_indices is None:
        start = prompt_index + 1
        if start >= len(sequence):
            print(f'{video_id}: no frames after the prompt. '
                  f'Set FRAME_INDICES = [{prompt_index}] to display the prompted frame.')
            return None
        frame_indices = np.linspace(
            start, len(sequence) - 1, num=min(3, len(sequence) - start), dtype=int).tolist()
    else:
        frame_indices = list(dict.fromkeys(frame_indices))
    if not frame_indices:
        print('FRAME_INDICES is empty; no comparison was drawn.')
        return None
    if any(not isinstance(index, (int, np.integer))
           or index < prompt_index or index >= len(sequence) for index in frame_indices):
        raise ValueError('FRAME_INDICES must contain integer indices from the prompt '
                         'through the last frame (forward propagation only).')
    ensure_comparison_masks(video_id, sequence, prompt, frame_indices)

    fig, axes = plt.subplots(len(frame_indices), 3, figsize=(12, 4 * len(frame_indices)),
                             squeeze=False)
    for row_index, frame_index in enumerate(frame_indices):
        source = sequence[frame_index]['image']
        with Image.open(ACTIVE_DATA_ROOT / source) as image:
            original = np.asarray(image.convert('RGB'))
        height, width = original.shape[:2]
        for column, title in enumerate(['Original', 'Pretrained', 'Fine-tuned']):
            axis = axes[row_index, column]
            axis.imshow(original)
            axis.set_title(
                f'Original\n{Path(source).name} | frame {frame_index}' if column == 0 else title,
                fontsize=11)
            if column > 0:
                model_name = ['pretrained', 'fine_tuned'][column - 1]
                path = RUN_DIR / 'video_masks' / model_name / video_id / f'{frame_index:08d}.png'
                with Image.open(path) as mask_image:
                    mask = np.asarray(mask_image) != 0
                overlay = np.zeros((*mask.shape, 4), dtype=float)
                overlay[mask] = [1.0, 0.0, 0.0, 0.35]
                axis.imshow(overlay)
                if mask.any():
                    axis.contour(np.arange(-1, width + 1), np.arange(-1, height + 1),
                                 np.pad(mask, 1), levels=[0.5], colors=['red'], linewidths=0.8)
            axis.set_xlim(-0.5, width - 0.5)
            axis.set_ylim(height - 0.5, -0.5)
            axis.set_aspect('equal')
            axis.axis('off')
    fig.suptitle(f'{video_id} | SAM 2.1 {MODEL_SIZE}', fontsize=14)
    fig.tight_layout(rect=(0, 0, 1, 0.96))
    output_path = RUN_DIR / f'segmentation_comparison_{video_id}.png'
    fig.savefig(output_path, dpi=160)
    plt.show()
    print('Saved:', output_path)
    return fig

In [ ]:
VIDEO_TO_SHOW = next(iter(test_sequences), None)
FRAME_INDICES = None  # Or specify frame indices, for example [1, 5, 9].

comparison_figure = show_segmentation_comparison(VIDEO_TO_SHOW, FRAME_INDICES)

## Use the checkpoint

In `SAM2-segmentation.ipynb`, set `MODEL_SIZE` to the same value used above and paste the `best.pt` path into `CUSTOM_CHECKPOINT`.
The cell below prints the matching model size and inference config. Use the upstream revision recorded in `best_metadata.json` for reproducible inference.
The inference notebook keeps its existing numeric JPEG filename convention.<br>
Each run saves full inference weights, configuration, video split/frame manifest, CSV metrics (UTF-8 BOM), and curves to Drive.
Run outputs can contain your source names and paths; keep them as local research outputs.

In [ ]:
print(f'MODEL_SIZE = {MODEL_SIZE!r}')
print(f'CUSTOM_CHECKPOINT = {str(BEST_PATH)!r}')
print(f'CONFIG = {INFERENCE_CONFIG!r}')
print('SAM2 upstream revision:', UPSTREAM_SHA)